# Case Study 3: AML Alert Screening Model

Audits a transaction-alert classifier (predicting whether a flagged transaction should be escalated to a SAR) across a synthetic `customer_segment` attribute — checking that escalation rates aren't disproportionately driven by segment rather than genuine risk signal. Synthetic data only.

In [1]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

from responsible_ai_finance import audit

pd.set_option("display.precision", 3)
np.random.seed(42)


In [2]:
rng = np.random.default_rng(23)
n = 1000

transaction_amount = rng.lognormal(mean=7, sigma=1.2, size=n)
velocity_24h = rng.poisson(2, n)
new_beneficiary = rng.integers(0, 2, n)
cross_border = rng.integers(0, 2, n)
customer_segment = rng.choice(["Retail", "SME"], size=n, p=[0.7, 0.3])

logit = (
    -3
    + np.log1p(transaction_amount) / 4
    + 0.6 * velocity_24h
    + 1.2 * new_beneficiary
    + 1.0 * cross_border
)
prob_escalate = 1 / (1 + np.exp(-logit))
y = (rng.uniform(0, 1, n) < prob_escalate).astype(int)

X = pd.DataFrame(
    {
        "transaction_amount": transaction_amount,
        "velocity_24h": velocity_24h,
        "new_beneficiary": new_beneficiary,
        "cross_border": cross_border,
    }
)
X_train, X_test, y_train, y_test, seg_train, seg_test = train_test_split(
    X, y, customer_segment, test_size=0.3, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
model.fit(X_train, y_train)
print(f"Test accuracy: {model.score(X_test, y_test):.3f}")
print(f"Escalation rate in test set: {y_test.mean():.3f}")


Test accuracy: 0.693
Escalation rate in test set: 0.700


In [3]:
report = audit(
    model,
    X_test.reset_index(drop=True),
    pd.Series(y_test).reset_index(drop=True),
    pd.Series(seg_test).reset_index(drop=True),
    model_name="aml-alert-screening-rf",
)
print(report.to_markdown())


# Responsible AI Audit — aml-alert-screening-rf
_Generated 2026-09-13T11:37:59.813501+00:00_

## Governance Flags
- ⚠️ Zeroing 'velocity_24h' flips 34.5% of predictions — check upstream data-quality guarantees for this feature.

## Fairness

- Demographic parity difference: **0.0081**
- Disparate impact ratio: **0.9898** (80% rule: PASS)
- Equalized odds — TPR difference: **0.0317**
- Equalized odds — FPR difference: **0.0926**

| Group | n | Selection rate | TPR | FPR |
|---|---|---|---|---|
| Retail | 201 | 0.796 | 0.857 | 0.630 |
| SME | 99 | 0.788 | 0.825 | 0.722 |

## Explainability

- Top features by mean |SHAP value|: new_beneficiary, cross_border, velocity_24h, transaction_amount
- SHAP local-fidelity MAE: **0.0000**

## Robustness

- Prediction flip rate under 5% Gaussian noise: **1.5%**
- Most fragile feature to dropout: **velocity_24h** (34.5% flip rate)


## Key Takeaways

- Financial crime models are audited for fairness too: an AML model that escalates one customer segment at a materially higher rate than its true risk warrants creates both a fair-treatment problem and, ironically, alert-fatigue that can mask genuine risk elsewhere.
- The robustness section matters operationally here: if zeroing `transaction_amount` (e.g. a missing/late-arriving field) flips a large share of predictions, that is a production data-quality dependency worth hardening before go-live.